# Dataset decision: Maven versus IBM
## 1. Evidence sources
IBM numbers are the existing conservative results in CONTEXT.md, not a retrained model. Maven numbers come from the compact handoff produced by notebook 04. Run 04 first. Verify input hashes before comparing. Different populations, prevalence and splits prevent direct claims that a higher raw AP or lower raw Brier proves superiority.

In [2]:
from pathlib import Path
import json,hashlib
import pandas as pd
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'CONTEXT.md').is_file() and (p/'datasets').is_dir())
path=ROOT/'notebooks'/'maven_evaluation_summary.json'
if not path.exists(): raise FileNotFoundError('Run notebook 04 first to create the evaluation summary.')
r=json.loads(path.read_text())
for name,digest in r['input_sha256'].items(): assert hashlib.sha256((ROOT/'datasets'/'CRM+Sales+Opportunities'/(name+'.csv')).read_bytes()).hexdigest()==digest
results=pd.DataFrame(r['test']).set_index('candidate')
val=pd.DataFrame(r['validation']).set_index('candidate')
chosen=r['chosen'];m=results.loc[chosen]
display(val.round(4));display(results.round(4))
comparison=pd.DataFrame({'IBM conservative':[77692,0.2252,0.6188,0.3105,0.1689,0.0319,0.033,0.425],'Maven locked':[r['closed_n'],r['test_prevalence'],m.ROC_AUC,m.PR_AUC_AP,m.Brier,m.Brier_skill,m.p_min,m.p_max]},index=['Usable closed population','Test WON prevalence','Test ROC-AUC','Test PR-AUC/AP','Test Brier','Brier skill vs prevalence','Probability minimum','Probability maximum'])
display(comparison.round(4))
print('Maven train/validation/test:',r['train_n'],r['validation_n'],r['test_n'],'| locked:',chosen)
print('IBM classification threshold 0.1844; Maven 0.5. Precision/recall cannot be compared without this context.')
display(pd.DataFrame(r['split']))
display(pd.DataFrame(r['calibration']).query('candidate == @chosen').round(4))
display(pd.DataFrame(r['history_support']))

,ROC_AUC,PR_AUC_AP,precision,recall,F1,Brier,p_min,p_max
candidate,,,,,,,,
A_generic / Prevalence,0.5000,0.5164,0.5164,1.0,0.6811,0.2722,0.6663,0.6663
B_history / Prevalence,0.5000,0.5164,0.5164,1.0,0.6811,0.2722,0.6663,0.6663
D_reduced / Prevalence,0.5000,0.5164,0.5164,1.0,0.6811,0.2722,0.6663,0.6663
C_product / Prevalence,0.5000,0.5164,0.5164,1.0,0.6811,0.2722,0.6663,0.6663
A_generic / Logistic Regression,0.5208,0.5316,0.5164,1.0,0.6811,0.2749,0.5697,0.7683
B_history / Logistic Regression,0.5015,0.5077,0.5164,1.0,0.6811,0.2991,0.5352,0.9422
D_reduced / Logistic Regression,0.4947,0.5033,0.5164,1.0,0.6811,0.3070,0.5617,0.9400
C_product / Logistic Regression,0.5083,0.5121,0.5164,1.0,0.6811,0.3137,0.5575,0.9544
A_generic / XGBoost,0.5143,0.5511,0.5164,1.0,0.6811,0.3300,0.6950,0.8633


,ROC_AUC,PR_AUC_AP,precision,recall,F1,Brier,p_min,p_max,ECE_10,Brier_skill
candidate,,,,,,,,,,
A_generic / Prevalence,0.5000,0.5978,0.5978,1.0,0.7483,0.2451,0.6663,0.6663,0.0685,0.0000
A_generic / Logistic Regression,0.5416,0.6258,0.5978,1.0,0.7483,0.2564,0.6043,0.8060,0.1312,-0.0459
A_generic / Random Forest,0.4881,0.6053,0.5978,1.0,0.7483,0.2926,0.6684,0.9198,0.2208,-0.1936
A_generic / XGBoost,0.4881,0.5921,0.5978,1.0,0.7483,0.2857,0.6824,0.8665,0.2072,-0.1655
B_history / Prevalence,0.5000,0.5978,0.5978,1.0,0.7483,0.2451,0.6663,0.6663,0.0685,0.0000
B_history / Logistic Regression,0.5018,0.6002,0.5978,1.0,0.7483,0.3011,0.6138,0.9729,0.2404,-0.2283
B_history / Random Forest,0.4859,0.5958,0.5978,1.0,0.7483,0.2982,0.7267,0.9051,0.2351,-0.2167
B_history / XGBoost,0.5064,0.6072,0.5978,1.0,0.7483,0.2909,0.7163,0.8951,0.2225,-0.1868
C_product / Prevalence,0.5000,0.5978,0.5978,1.0,0.7483,0.2451,0.6663,0.6663,0.0685,0.0000


,IBM conservative,Maven locked
Usable closed population,77692.0000,6711.0000
Test WON prevalence,0.2252,0.5978
Test ROC-AUC,0.6188,0.5000
Test PR-AUC/AP,0.3105,0.5978
Test Brier,0.1689,0.2451
Brier skill vs prevalence,0.0319,0.0000
Probability minimum,0.0330,0.6663
Probability maximum,0.4250,0.6663


Maven train/validation/test: 2679 730 1380 | locked: A_generic / Prevalence
IBM classification threshold 0.1844; Maven 0.5. Precision/recall cannot be compared without this context.


,partition,rows,won_rate,engage_min,engage_max,close_max,no_account_history,account_history_lt5
0,train,2679,0.6662933930571109,2016-10-20,2017-06-26,2017-06-30,1079,1417
1,validation,730,0.5164383561643836,2017-07-01,2017-08-30,2017-08-31,0,0
2,test,1380,0.5978260869565217,2017-09-01,2017-10-31,2017-12-31,0,0


,candidate,probability_bin,n,predicted,observed
0,A_generic / Prevalence,0–10%,0,NaN,NaN
1,A_generic / Prevalence,10–20%,0,NaN,NaN
2,A_generic / Prevalence,20–30%,0,NaN,NaN
3,A_generic / Prevalence,30–40%,0,NaN,NaN
4,A_generic / Prevalence,40–50%,0,NaN,NaN
5,A_generic / Prevalence,50–60%,0,NaN,NaN
6,A_generic / Prevalence,60–70%,1380,0.6663,0.5978
7,A_generic / Prevalence,70–80%,0,NaN,NaN
8,A_generic / Prevalence,80–90%,0,NaN,NaN
9,A_generic / Prevalence,90–100%,0,NaN,NaN


,partition,account_history,n,won_rate
0,train,none,1079,0.725672
1,train,1–4,338,0.686391
2,train,5+,1262,0.610143
3,validation,none,0,NaN
4,validation,1–4,0,NaN
5,validation,5+,730,0.516438
6,test,none,0,NaN
7,test,1–4,0,NaN
8,test,5+,1380,0.597826


## 2. Structural comparison
| Dimension | IBM | Maven |
|---|---|---|
| Interpretable inputs | Narrow product/geography/route; opaque customer bands | Sector, documented revenue in millions USD, headcount, age, list price |
| Engineering | Mostly transforms/interactions; no customer ledger | Strictly prior account/agent/product outcomes, age/logs/ratios |
| Timing certainty | No snapshot/calendar dates | Engage/close dates establish event ordering; reference fields still undated |
| Identity | Repeated opportunity IDs, no customer ID | 85 account names, 30 active agents, 7 products |
| Temporal evaluation | Grouped random holdout | Calendar holdout with observable training/validation labels |
| Provenance | IBM-labelled file; exact lineage/reuse unresolved | Named Maven source; fictitious hardware company, realism/generalization limited; verify terms |
| Leakage | Progression/duration fields; unknown timing | Current stage/value/date excluded; history requires strict cutoffs and outcome availability |
| Memorization | Product/domain specificity | Exact finances fingerprint accounts; price fingerprints product; name removal insufficient |
| Open population | Resolved-only export | 2,089 open retained; 501 currently satisfy engagement scoring contract |
| Two modes | No verified historical availability states | Observable history counts, but late cohorts may lack no-history records |
| React/PostgreSQL | Requires inventing new persistent account/history structures | Natural account/product/agent/opportunity/prediction/outcome entities |
| Generalization | Not validated beyond historical dataset | Not validated to new firms/agents/products or real-world sales |

A stronger relational schema is useful for software engineering, but does not establish useful win-probability discrimination. The ~0.82 IBM enriched result is validation-only and depends on unresolved feature timing; do not compare it to Maven's test result as a model contest.

## 3. Decision evidence and proposed scope
Use the executed results, history ablations and baseline-relative probability quality. Do not choose a final dataset based on schema or raw class-prevalence-dependent metrics alone. The exact Maven output means: probability of eventual WON among historically resolved, engaged opportunities with known account/product/agent, conditional on static-snapshot and complete-ledger assumptions. It is not expected revenue, causal seller effectiveness, or benefit from an action. A new-account claim would require an additional entity-held-out study.

Provisional title: **Leakage-Aware Sales Opportunity Probability Estimation with Point-in-Time CRM History**. A future React/PostgreSQL workflow can import account/product/agent records, record engagement, compute immutable as-of snapshots, display probability and support counts, log human actions, and record outcomes. Never ask users to manually type historical win rates; derive them from the ledger. No React application is edited here.

## Final recommendation — CONTINUE TESTING
**Do not select Maven for individualized win scoring on present evidence; do not automatically select IBM either.** Its relational structure is better, but all learned candidates—including the executed XGBoost follow-up—lose to training prevalence on validation and test Brier.

### Evidence
- 6,711 usable closed opportunities, 63.15% Won overall. Fit/validation/test: **2,679 / 730 / 1,380**, with observed win rates **66.63% / 51.64% / 59.78%**.
- Validation selects prevalence: constant **66.63%**, test AUC **0.5000**, Brier **0.2451**, calibration gap **6.85 percentage points**. No individualized ranking.
- Best nonconstant validation-Brier challenger remains generic Logistic Regression: test AUC **0.5416**, AP **0.6258**, Brier **0.2564**, range **60.43–80.60%**, ECE **13.12 points**.
- XGBoost test AUC for A/B/C/D: **0.4881 / 0.5064 / 0.5113 / 0.5128**. Brier: **0.2857 / 0.2909 / 0.2932 / 0.2908**. ECE is **20.72–22.87 points**; all predict Won at 0.5. More complex modelling does not fix poor ranking/calibration here.
- RF A→B history test AUC **0.4881→0.4859**, Brier **0.2926→0.2982**. Identity additions and frozen-ledger checks do not establish stable signal.
- Every validation/test record has ≥5 previous account closures. Limited-history routing is unvalidated. Early missing history reflects the truncated ledger, not proven new accounts.
- Label observability excludes 970 pre-July engagements from training and 605 July–August closed engagements from validation. Another 206 eligible validation engagements remain open in the extract; the test cohort excludes 43 unresolved opportunities. Calendar coverage and closed-only selection limit conclusions.
- IBM conservative AUC **0.6188** and Brier skill **3.19%** are stronger on its own evaluation, but its grouped random split and timing/provenance gaps remain. Higher Maven AP mostly reflects majority-win prevalence; raw cross-dataset Brier/AP comparisons are not decisive.

### Project scope and features
**Provisional title:** Leakage-Aware Sales Opportunity Probability Estimation with Point-in-Time CRM History.
**No deployment feature set approved.** Reduced research candidates: sector, approximate company age, log revenue/headcount, log revenue per employee, subsidiary indicator and strict historical account/agent counts/rates with missingness indicators. Calendar features and entity-fingerprinting financials require ablations on new development evidence.
**Exclude:** current stage, close date/value, ID and current duration. Default-exclude exact account/agent/manager/region names and product/list-price proxies. Reduce redundant raw/log and count/win/rate combinations only using development evaluation.
**Two models:** not justified. Prefer one history-aware model if subsequent evidence supports it.
**Probability meaning:** eventual WON among engaged, historically resolved opportunities with known account/product/agent, conditional on undated-reference and ledger-availability assumptions; not expected revenue, causal action benefit or individual certainty.

### Next evaluation gate
Investigate prespecified maturation windows/rolling origins and non-calendar signal before aggressive tuning or application design. Calibration may reduce overconfidence but cannot manufacture ranking. The XGBoost extension reused the already inspected test period; it is an exploratory follow-up, not a fresh untouched holdout.

### Execution
CPU-only **XGBoost 3.4.1** was installed in the existing environment at the user's request. All nine modelling code cells were rerun; 16 feature-set/model comparisons and their calibration outputs are retained. Fixed pre-existing XGBoost settings: 150 trees, depth 2, learning rate 0.04, min_child_weight 15, reg_lambda 10, subsample/colsample 0.9, histogram trees, seed 42. No tuning, random split, test calibration fitting, or two-mode deployment. The JSON handoff records versions/parameters and hashes; this notebook refreshes from it. IBM/source CSVs remain unchanged; no React changes, commits or pushes.